# Tính năng 5 – Trang **Đánh giá mô hình** (phần 2: độ tin cậy và chuẩn production)

Bốn câu hỏi mà một con số MAE không trả lời được:
1. **Hiệu chuẩn:** nói "80%" thì có đúng 80%?
2. **Bootstrap:** mô hình thắng thật hay nhờ may?
3. **Công bằng:** có đoán lệch có hệ thống cho nhóm nào? Còn giới tính?
4. **Chuẩn production:** đã đủ tốt để đưa cho người dùng thật chưa?

Mã nguồn: `salary_validation.py`, `production_standard.py`, `prediction_support.py`, `model_promotion.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.data.dataset import load_dataset
from career_market.model.estimators import MODEL_LABELS, feature_frame
from career_market.model.training import load_model, load_production
from career_market.model.validation import interval_ratios, load_validation

frame = load_dataset()
validation = load_validation()
production = load_production()
model, name, report = load_model()
valid = frame[frame["split"] == "valid"].reset_index(drop=True)
test = frame[frame["split"] == "test"].reset_index(drop=True)
valid_pred = model.predict(feature_frame(valid))
test_pred = model.predict(feature_frame(test))
print(MODEL_LABELS[name], "| valid", len(valid), "| test", len(test), "| công ty test", test["company_key"].nunique())

## 1. Hiệu chuẩn – "dự báo thời tiết 80%"
Nếu đài nói "80% khả năng mưa" cho 100 ngày, đài tốt phải có ~80 ngày mưa thật. Khoảng 80% của ứng dụng = dự đoán × [p10; p90] của tỉ lệ *lương thật / dự đoán* trên valid; kiểm tra trên test chưa dùng. Dùng **tỉ lệ** chứ không ± số cố định vì sai số với người lương 80 triệu lớn hơn với người 15 triệu.

In [ ]:
low, high = interval_ratios(valid["salary"].to_numpy(), valid_pred, 0.8)
inside = (test["salary"] >= test_pred * low) & (test["salary"] <= test_pred * high)
print(f"Khoảng 80% = dự đoán × [{low:.2f} ; {high:.2f}] → bao phủ {inside.mean():.1%} lương thật trên test")
display(pd.DataFrame(validation["calibration"]["levels"])[["level", "coverage", "mean_width"]].rename(
    columns={"level": "Tuyên bố", "coverage": "Thực tế", "mean_width": "Độ rộng TB (triệu)"}))
display(pd.DataFrame(validation["calibration"]["by_salary_band"]).rename(columns={"group": "Nhóm lương", "n": "Số tin", "coverage": "Bao phủ 80%"}))
charts.reliability(validation["calibration"]);

Khoảng hơi **rộng hơn cần** (thận trọng) ở mức chung, nhưng chỉ đúng ~72% ở nhóm lương cao.

## 2. Bootstrap theo công ty – "thi lại 1.000 lần với đề khác"
Rút ngẫu nhiên có hoàn lại các **công ty** của tập test (không rút từng tin vì tin cùng công ty giống nhau), chấm lại 1.000 lần. So **theo cặp** trên cùng một đề.

In [ ]:
boot = validation["bootstrap"]
display(pd.DataFrame({MODEL_LABELS[n]: {"MAE": v["MAE"]["point"], "KTC 95% thấp": v["MAE"]["ci95"][0], "KTC 95% cao": v["MAE"]["ci95"][1]}
                      for n, v in boot["models"].items()}).T)
display(pd.DataFrame({MODEL_LABELS[n]: {"Chênh MAE TB": d["mean"], "KTC thấp": d["ci95"][0], "KTC cao": d["ci95"][1], "Tỉ lệ GB tốt hơn": d["p_better"]}
                      for n, d in boot["differences"].items()}).T)
charts.bootstrap_differences(boot, MODEL_LABELS);

Gradient Boosting thắng Ridge và Random Forest ở khoảng 85–88% số đề, nhưng khoảng 95% của chênh lệch **vẫn chứa 0** → trên 288 tin thật, **chưa đủ bằng chứng** để khẳng định nó tốt hơn hẳn. (Trên dữ liệu cũ lẫn tin sinh, con số này là 100% – đẹp giả.) Với mốc trung vị thì thắng chắc chắn.

## 3. Công bằng theo nhóm
Chiếc cân lệch 1 kg với mọi người vẫn "công bằng"; chỉ lệch với người thấp là thiên vị. Độ lệch nhóm = trung bình (dự đoán − thật) / thật; khoảng tin cậy chứa 0 → chưa có bằng chứng lệch; nhóm < 20 tin chỉ báo cáo.

In [ ]:
def fairness_table(key):
    g = pd.DataFrame(validation["fairness"][key]["groups"])
    g["KTC 95%"] = g["bias_pct_ci95"].map(lambda c: f"{c[0]:+.1f} → {c[1]:+.1f}")
    g["Kết luận"] = np.where(~g["enough"], "ít mẫu – không kết luận",
                    np.where(g["bias_pct_ci95"].map(lambda c: c[0] > 0), "đoán CAO có hệ thống",
                    np.where(g["bias_pct_ci95"].map(lambda c: c[1] < 0), "đoán THẤP có hệ thống", "không có bằng chứng lệch")))
    return g[["group", "n", "MAPE", "bias_pct", "KTC 95%", "Kết luận"]].rename(columns={"group": validation["fairness"][key]["label"], "n": "Số tin", "bias_pct": "Độ lệch %"})

print(f"Độ lệch toàn mô hình: {((test_pred - test['salary']) / test['salary']).mean() * 100:+.1f}% (hệ quả của việc học trên logarit)")
display(fairness_table("region"))
display(fairness_table("level"))
charts.fairness_bias(validation["fairness"]["level"]);

In [ ]:
display(fairness_table("platform"))
gender = validation["gender"]
print(f"Giới tính: dữ liệu không có giới tính người được tuyển, mô hình không dùng giới tính. Tin có điều kiện giới tính: {gender['mentioned']} / {len(frame):,}")
pd.DataFrame(gender["examples"])[["title", "source", "gender_requirement", "salary"]]

**Junior bị đoán cao ~7%** (khoảng tin cậy không chứa 0): người mới đi làm nên trừ hao. Giới tính **không đo được** – nói "mô hình công bằng giới" sẽ không trung thực; điều kiện giới tính trong tin là dấu hiệu phân biệt đối xử theo Bộ luật Lao động 2019.

## 4. Chuẩn production – theo cách Glassdoor, LinkedIn công bố
Ba chỉ số dễ hiểu: **trung vị sai số %**, **tỉ lệ dự đoán sai ≤ 20%**, **độ phủ khoảng p10–p90**. Ngưỡng chung so bằng **cận bất lợi** của khoảng bootstrap (không "đậu nhờ may"); ngưỡng nhóm so bằng giá trị điểm.

In [ ]:
display(pd.DataFrame(production["thresholds"]).T)
rows = [{"Chỉ số": k, "Giá trị": m["point"], "KTC 95%": f"{m['ci95'][0]:.1f} – {m['ci95'][1]:.1f}", "Ngưỡng": m["threshold"], "Đạt": m["pass"]}
        for k, m in production["overall"]["metrics"].items()]
display(pd.DataFrame(rows))
groups = [{"Phạm vi": f"{dim}: {g['group']}", "Số tin test": g["n"], "Kết quả": g["status"],
           **{k: round(m["point"], 1) for k, m in g.get("metrics", {}).items()}}
          for dim in ("role", "level") for g in production[dim] if g["status"] != "not_scored"]
display(pd.DataFrame(groups))
print("Đạt chuẩn production:", production["production_ready"], "| nhóm trượt:", production["failing"])

**Mô hình chưa đạt chuẩn.** Vì vậy ứng dụng: (1) ghi rõ "kết quả mang tính tham khảo"; (2) chỉ hiện **con số** khi nhóm vai trò × cấp bậc có ≥ 20 tin train và không trượt chuẩn, còn lại chỉ hiện **khoảng mở rộng**; (3) khi huấn luyện lại, mô hình mới chỉ thay mô hình cũ nếu qua **cổng** (đạt chuẩn, hoặc cả hai chưa đạt mà mô hình mới không tệ hơn ở chỉ số chung nào và qua nhiều phạm vi hơn / tốt hơn hẳn ở ít nhất một chỉ số).

In [ ]:
support = report["support"]["cohort_train_rows"]
cohorts = pd.Series(support, name="Số tin train").sort_values(ascending=False)
print(f"{(cohorts >= 20).sum()} / {len(cohorts)} nhóm vai trò × cấp bậc đủ 20 tin train để hiện con số")
cohorts.head(12).to_frame()